# Proxy-Shuffle Negative Control

Runs the full 20-seed real-data proxy-shuffle negative control and saves summary tables only.

In [1]:
from pathlib import Path
import os
import shutil
import sys

import pandas as pd
from IPython.display import display

os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

current = Path.cwd().resolve()
repo_root = next((p for p in [current, *current.parents] if (p / "experiments").exists() and (p / "config").exists()), None)
if repo_root is None:
    raise RuntimeError(f"Could not locate repo root from {current}")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from evaluation.stratified_kstar import build_economics_stratifiers, build_energy_stratifiers
from experiments.run_complete_20seed_suite import SEEDS as COMPLETE_SEEDS
from experiments.run_proxy_shuffle_negative_control import _run_economics, _run_energy
from experiments.run_proxy_shuffle_stratified_kstar import ECON_RAW, ENERGY_RAW, ORIGINAL_ECON_ROOT, ORIGINAL_ENERGY_ROOT, _run_domain

SEEDS = list(range(20))
OUTPUT_ROOT = repo_root / "outputs" / "negative_controls" / "proxy_shuffle_20seed_20260511"
COMPARISON_DIR = OUTPUT_ROOT / "comparison"
FORCE = False
DRY_RUN = False
SMOKE = False
PROXY_PERM_SEED_OFFSET = 10000
N_PERM = 2000
BOOTSTRAP_SAMPLES = 10000
RUN_PROXY_SHUFFLE = True
RUN_STRATIFIED_AUDIT = True

invalid = [seed for seed in SEEDS if seed not in COMPLETE_SEEDS]
if invalid:
    raise ValueError(f"Seeds outside the supported complete-suite range: {invalid}")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
COMPARISON_DIR.mkdir(parents=True, exist_ok=True)

def clean_incomplete_proxy_runs() -> None:
    specs = [
        (OUTPUT_ROOT / "economics" / "cmdl", "economics_proxy_shuffle_seed"),
        (OUTPUT_ROOT / "energy" / "cmdl", "energy_proxy_shuffle_seed"),
    ]
    for root, prefix in specs:
        if not root.exists():
            continue
        for seed in SEEDS:
            run_dir = root / f"{prefix}{seed}"
            if run_dir.exists() and not (run_dir / "summary.json").exists():
                print(f"[clean] incomplete artifact: {run_dir}")
                shutil.rmtree(run_dir)

clean_incomplete_proxy_runs()
display(pd.Series({"seeds": SEEDS, "output_root": OUTPUT_ROOT, "force": FORCE, "smoke": SMOKE, "n_perm": N_PERM}).to_frame("value"))

c:\Users\42155\anaconda3\envs\PTenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,value
seeds,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,..."
output_root,C:\DevSpace\PyDevspace\CMDL\outputs\negative_c...
force,False
smoke,False
n_perm,2000


In [2]:
if RUN_PROXY_SHUFFLE:
    _run_economics(OUTPUT_ROOT, SEEDS, force=FORCE, dry_run=DRY_RUN, seed_offset=PROXY_PERM_SEED_OFFSET, smoke=SMOKE)
    _run_energy(OUTPUT_ROOT, SEEDS, force=FORCE, dry_run=DRY_RUN, seed_offset=PROXY_PERM_SEED_OFFSET, smoke=SMOKE)
else:
    print("RUN_PROXY_SHUFFLE = False; using existing proxy-shuffle artifacts if present.")

[run] economics proxy-shuffle AC-GATE seed 0: C:\DevSpace\PyDevspace\CMDL\outputs\negative_controls\proxy_shuffle_20seed_20260511\economics\cmdl\economics_proxy_shuffle_seed0
[economics_proxy_shuffle_seed0] epoch=001 train_total=1.9567 val_task=2.9869 val_r2=-0.0791 val_proxy_r2=-0.3824
[economics_proxy_shuffle_seed0] epoch=010 train_total=0.6933 val_task=2.4865 val_r2=0.1121 val_proxy_r2=-0.3327
[economics_proxy_shuffle_seed0] epoch=020 train_total=0.5186 val_task=2.4908 val_r2=0.0914 val_proxy_r2=-0.2790
[economics_proxy_shuffle_seed0] epoch=030 train_total=0.4035 val_task=2.4756 val_r2=0.0963 val_proxy_r2=-0.2217
[economics_proxy_shuffle_seed0] early stopping at epoch 37
[run] economics proxy-shuffle AC-GATE seed 1: C:\DevSpace\PyDevspace\CMDL\outputs\negative_controls\proxy_shuffle_20seed_20260511\economics\cmdl\economics_proxy_shuffle_seed1
[economics_proxy_shuffle_seed1] epoch=001 train_total=1.0464 val_task=2.9822 val_r2=-0.0695 val_proxy_r2=-0.2972
[economics_proxy_shuffle_seed

In [3]:
summaries = []
tables = {}
if RUN_STRATIFIED_AUDIT:
    econ_per_seed, econ_aggregated, econ_summary = _run_domain(
        "economics",
        ORIGINAL_ECON_ROOT,
        OUTPUT_ROOT,
        ECON_RAW,
        build_economics_stratifiers,
        COMPARISON_DIR,
        n_perm=N_PERM,
        bootstrap_samples=BOOTSTRAP_SAMPLES,
    )
    energy_per_seed, energy_aggregated, energy_summary = _run_domain(
        "energy",
        ORIGINAL_ENERGY_ROOT,
        OUTPUT_ROOT,
        ENERGY_RAW,
        build_energy_stratifiers,
        COMPARISON_DIR,
        n_perm=N_PERM,
        bootstrap_samples=BOOTSTRAP_SAMPLES,
    )
    combined_summary = pd.concat([econ_summary, energy_summary], ignore_index=True, sort=False)
    combined_summary.to_csv(COMPARISON_DIR / "proxy_shuffle_summary.csv", index=False)
    tables = {
        "economics_proxy_shuffle_stratified_kstar_per_seed": econ_per_seed,
        "economics_proxy_shuffle_stratified_kstar_aggregated": econ_aggregated,
        "economics_proxy_shuffle_summary": econ_summary,
        "energy_proxy_shuffle_stratified_kstar_per_seed": energy_per_seed,
        "energy_proxy_shuffle_stratified_kstar_aggregated": energy_aggregated,
        "energy_proxy_shuffle_summary": energy_summary,
        "proxy_shuffle_summary": combined_summary,
    }
else:
    print("RUN_STRATIFIED_AUDIT = False; using existing comparison tables if present.")
    for path in COMPARISON_DIR.glob("*.csv"):
        tables[path.stem] = pd.read_csv(path)

pd.Series({name: len(frame) for name, frame in tables.items()}, name="rows").to_frame()


=== economics proxy-shuffle paired summary ===
   domain                   stratifier  n_pairs  original_abs_rho_mean  proxy_shuffled_abs_rho_mean  delta_abs_rho_mean  delta_abs_rho_ci_low  delta_abs_rho_ci_high  paired_wilcoxon_original_gt_proxy_p  original_seed_p_lt_05_share  proxy_shuffled_seed_p_lt_05_share  original_fisher_p  proxy_shuffled_fisher_p  original_kstar_std_mean  proxy_shuffled_kstar_std_mean  original_test_r2_mean  proxy_shuffled_test_r2_mean  delta_test_r2_mean
economics                hc_mean_train       20               0.370599                     0.066889            0.303710              0.220950               0.387223                         1.907349e-06                         0.80                                0.0       1.009779e-46                 0.936566                 0.168247                       0.183879               0.054146                     0.054963           -0.000818
economics log_capital_per_worker_train       20               0.256791      

,rows
economics_proxy_shuffle_stratified_kstar_per_seed,120
economics_proxy_shuffle_stratified_kstar_aggregated,6
economics_proxy_shuffle_summary,3
energy_proxy_shuffle_stratified_kstar_per_seed,120
energy_proxy_shuffle_stratified_kstar_aggregated,6
energy_proxy_shuffle_summary,3
proxy_shuffle_summary,6


In [4]:
for name, frame in tables.items():
    print(f"\n=== {name} ({len(frame)} rows) ===")
    display(frame.head(20))


=== economics_proxy_shuffle_stratified_kstar_per_seed (120 rows) ===


,method,seed,stratifier,n_entities,spearman_rho,perm_p_two_sided,kstar_std,degenerate
0,AC-GATE Original,0,log_gdp_per_worker_train,88,-0.603977,0.0000,0.033778,False
1,AC-GATE Original,0,hc_mean_train,88,-0.806224,0.0000,0.033778,False
2,AC-GATE Original,0,log_capital_per_worker_train,88,-0.564844,0.0000,0.033778,False
3,AC-GATE Original,1,log_gdp_per_worker_train,88,-0.348772,0.0020,0.022675,False
4,AC-GATE Original,1,hc_mean_train,88,-0.406273,0.0000,0.022675,False
5,AC-GATE Original,1,log_capital_per_worker_train,88,-0.279508,0.0110,0.022675,False
6,AC-GATE Original,2,log_gdp_per_worker_train,88,-0.424747,0.0000,0.224089,False
7,AC-GATE Original,2,hc_mean_train,88,-0.508453,0.0000,0.224089,False
8,AC-GATE Original,2,log_capital_per_worker_train,88,-0.416769,0.0000,0.224089,False
9,AC-GATE Original,3,log_gdp_per_worker_train,88,0.017911,0.8785,0.028368,False



=== economics_proxy_shuffle_stratified_kstar_aggregated (6 rows) ===


,method,stratifier,n_seeds_total,n_seeds_valid,rho_mean,rho_median,abs_rho_mean,share_seeds_p_lt_05,share_seeds_p_lt_01,fisher_combined_p
0,AC-GATE Original,hc_mean_train,20,20,-0.141652,-0.295865,0.370599,0.80,0.75,1.009779e-46
1,AC-GATE Original,log_capital_per_worker_train,20,20,-0.099347,-0.135210,0.256791,0.65,0.50,1.382788e-24
2,AC-GATE Original,log_gdp_per_worker_train,20,20,-0.109463,-0.146094,0.278268,0.70,0.55,3.494823e-37
3,AC-GATE Proxy Shuffled,hc_mean_train,20,20,-0.004915,-0.010734,0.066889,0.00,0.00,9.365659e-01
4,AC-GATE Proxy Shuffled,log_capital_per_worker_train,20,20,0.003675,0.005415,0.050903,0.00,0.00,9.957766e-01
5,AC-GATE Proxy Shuffled,log_gdp_per_worker_train,20,20,-0.007037,-0.001655,0.054840,0.00,0.00,9.915230e-01



=== economics_proxy_shuffle_summary (3 rows) ===


...

,method,stratifier,n_seeds_total,n_seeds_valid,rho_mean,rho_median,abs_rho_mean,share_seeds_p_lt_05,share_seeds_p_lt_01,fisher_combined_p
0,AC-GATE Original,government_effectiveness_train,20,20,0.014916,0.129111,0.715528,0.90,0.85,9.034017e-77
1,AC-GATE Original,log_gdp_per_capita_train,20,20,0.011460,0.185157,0.608751,0.90,0.85,1.452259e-77
2,AC-GATE Original,rule_of_law_train,20,20,0.021033,0.236864,0.734692,0.95,0.90,1.788747e-79
3,AC-GATE Proxy Shuffled,government_effectiveness_train,20,20,0.016123,-0.002668,0.086359,0.10,0.00,6.006019e-01
4,AC-GATE Proxy Shuffled,log_gdp_per_capita_train,20,20,0.021167,-0.006462,0.079115,0.05,0.00,7.522986e-01
5,AC-GATE Proxy Shuffled,rule_of_law_train,20,20,0.012904,-0.019651,0.097064,0.05,0.00,3.874395e-01



=== energy_proxy_shuffle_summary (3 rows) ===


,domain,stratifier,n_pairs,original_abs_rho_mean,proxy_shuffled_abs_rho_mean,delta_abs_rho_mean,delta_abs_rho_ci_low,delta_abs_rho_ci_high,paired_wilcoxon_original_gt_proxy_p,original_seed_p_lt_05_share,proxy_shuffled_seed_p_lt_05_share,original_fisher_p,proxy_shuffled_fisher_p,original_kstar_std_mean,proxy_shuffled_kstar_std_mean,original_test_r2_mean,proxy_shuffled_test_r2_mean,delta_test_r2_mean
0,energy,government_effectiveness_train,20,0.715528,0.086359,0.629169,0.497571,0.747046,1.907349e-06,0.90,0.10,9.034017e-77,0.600602,0.10861,0.073245,-0.028567,-0.028024,-0.000544
1,energy,log_gdp_per_capita_train,20,0.608751,0.079115,0.529636,0.423837,0.624155,9.536743e-07,0.90,0.05,1.452259e-77,0.752299,0.10861,0.073245,-0.028567,-0.028024,-0.000544
2,energy,rule_of_law_train,20,0.734692,0.097064,0.637628,0.523218,0.739708,9.536743e-07,0.95,0.05,1.788747e-79,0.387440,0.10861,0.073245,-0.028567,-0.028024,-0.000544



=== proxy_shuffle_summary (6 rows) ===


,domain,stratifier,n_pairs,original_abs_rho_mean,proxy_shuffled_abs_rho_mean,delta_abs_rho_mean,delta_abs_rho_ci_low,delta_abs_rho_ci_high,paired_wilcoxon_original_gt_proxy_p,original_seed_p_lt_05_share,proxy_shuffled_seed_p_lt_05_share,original_fisher_p,proxy_shuffled_fisher_p,original_kstar_std_mean,proxy_shuffled_kstar_std_mean,original_test_r2_mean,proxy_shuffled_test_r2_mean,delta_test_r2_mean
0,economics,hc_mean_train,20,0.370599,0.066889,0.303710,0.220950,0.387223,1.907349e-06,0.80,0.00,1.009779e-46,0.936566,0.168247,0.183879,0.054146,0.054963,-0.000818
1,economics,log_capital_per_worker_train,20,0.256791,0.050903,0.205889,0.154416,0.259563,9.536743e-07,0.65,0.00,1.382788e-24,0.995777,0.168247,0.183879,0.054146,0.054963,-0.000818
2,economics,log_gdp_per_worker_train,20,0.278268,0.054840,0.223429,0.168255,0.281591,4.768372e-06,0.70,0.00,3.494823e-37,0.991523,0.168247,0.183879,0.054146,0.054963,-0.000818
3,energy,government_effectiveness_train,20,0.715528,0.086359,0.629169,0.497571,0.747046,1.907349e-06,0.90,0.10,9.034017e-77,0.600602,0.108610,0.073245,-0.028567,-0.028024,-0.000544
4,energy,log_gdp_per_capita_train,20,0.608751,0.079115,0.529636,0.423837,0.624155,9.536743e-07,0.90,0.05,1.452259e-77,0.752299,0.108610,0.073245,-0.028567,-0.028024,-0.000544
5,energy,rule_of_law_train,20,0.734692,0.097064,0.637628,0.523218,0.739708,9.536743e-07,0.95,0.05,1.788747e-79,0.387440,0.108610,0.073245,-0.028567,-0.028024,-0.000544
